# Binning and Discretization

**Discretization** (commonly called **Binning**) is the process of transforming continuous numerical variables into discrete categorical "bins" or "buckets." 

Why would we want to throw away the exact numerical detail?
1. **Business Logic**: Sometimes the exact number doesn't matter. A marketing team doesn't care if a customer is exactly 22 or 23; they just want to target the "18-25 Young Adult" demographic.
2. **Handling Outliers**: If you put a massive outlier into a "High Income" bin, its extreme numerical value can no longer drag the math of the model around.
3. **Non-Linear Relationships**: Sometimes the relationship between a feature and a target isn't a straight line. Binning helps linear models capture complex patterns.

Let's set up a Python sandbox to see how we create these buckets!

In [1]:
import pandas as pd
import numpy as np

# Create a dataset of customers with exact ages and incomes
data = {
    'customer_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'age': [15, 22, 24, 35, 42, 55, 68, 81],
    'income': [5000, 35000, 40000, 85000, 120000, 45000, 55000, 250000]
}

df = pd.DataFrame(data)

print("--- Original Continuous Data ---")
display(df)

--- Original Continuous Data ---


,customer_id,age,income
0,1,15,5000
1,2,22,35000
2,3,24,40000
3,4,35,85000
4,5,42,120000
5,6,55,45000
6,7,68,55000
7,8,81,250000


# 1. Custom Binning with Pandas (`pd.cut`)
The most common way to bin data is to define your own exact boundaries. You use `pd.cut()` when you have specific **business rules** you need to follow.

Let's group our ages into standard marketing demographics: Child (0-17), Young Adult (18-35), Adult (36-60), and Senior (61+).

In [2]:
# Create a copy
df_cut = df.copy()

# 1. Define the exact edges of the bins
# Note: We start at 0 and end at infinity (np.inf) to catch everything
age_bins = [0, 17, 35, 60, np.inf]

# 2. Define the labels for the bins
age_labels = ['Child', 'Young Adult', 'Adult', 'Senior']

# 3. Apply pd.cut()
df_cut['age_group'] = pd.cut(df_cut['age'], bins=age_bins, labels=age_labels)

print("--- Data after Custom Binning (pd.cut) ---")
display(df_cut[['customer_id', 'age', 'age_group']])

--- Data after Custom Binning (pd.cut) ---


,customer_id,age,age_group
0,1,15,Child
1,2,22,Young Adult
2,3,24,Young Adult
3,4,35,Young Adult
4,5,42,Adult
5,6,55,Adult
6,7,68,Senior
7,8,81,Senior


# 2. Quantile Binning with Pandas (`pd.qcut`)
What if you don't know where the boundaries should be? What if you just want to split your customers into "Low", "Medium", and "High" income brackets, and you want roughly the same number of people in each bucket?

You use `pd.qcut()` (Quantile Cut). Instead of defining exact numerical edges, you tell Pandas how many buckets you want, and it will figure out the boundaries to ensure the buckets are equally filled.

In [3]:
# Apply pd.qcut() to split income into 3 equally-sized groups (tertiles)
df_cut['income_bracket'] = pd.qcut(df_cut['income'], q=3, labels=['Low', 'Medium', 'High'])

print("--- Data after Quantile Binning (pd.qcut) ---")
display(df_cut[['customer_id', 'income', 'income_bracket']])

# Let's verify that the buckets are roughly equal in size!
print("\n--- Bucket Counts ---")
print(df_cut['income_bracket'].value_counts())

--- Data after Quantile Binning (pd.qcut) ---


,customer_id,income,income_bracket
0,1,5000,Low
1,2,35000,Low
2,3,40000,Low
3,4,85000,High
4,5,120000,High
5,6,45000,Medium
6,7,55000,Medium
7,8,250000,High



--- Bucket Counts ---
income_bracket
Low       3
High      3
Medium    2
Name: count, dtype: int64


# 3. Binning for Machine Learning (`KBinsDiscretizer`)
Just like Scaling and Categorical Encoding, if you are building a formal Machine Learning pipeline, you should use Scikit-Learn instead of Pandas. 

Scikit-Learn's `KBinsDiscretizer` handles new, unseen data perfectly and can even One-Hot Encode the bins automatically in a single step!

In [4]:
from sklearn.preprocessing import KBinsDiscretizer

# 1. Initialize the Discretizer
# n_bins=3: We want 3 buckets
# encode='ordinal': Returns 0, 1, 2 (Use 'onehot' for dummy variables!)
# strategy='quantile': Works exactly like pd.qcut. (Use 'uniform' to space bins evenly by value).
discretizer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')

# Create a copy
df_sklearn = df.copy()

# 2. Fit and Transform the income column
# Note: Scikit-Learn expects a 2D array, so we pass df[['income']]
df_sklearn['income_encoded_bin'] = discretizer.fit_transform(df_sklearn[['income']])

print("--- Scikit-Learn KBinsDiscretizer Output ---")
display(df_sklearn[['customer_id', 'income', 'income_encoded_bin']])

--- Scikit-Learn KBinsDiscretizer Output ---


/root/micromamba/envs/ds/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,customer_id,income,income_encoded_bin
0,1,5000,0.0
1,2,35000,0.0
2,3,40000,0.0
3,4,85000,2.0
4,5,120000,2.0
5,6,45000,1.0
6,7,55000,1.0
7,8,250000,2.0


*(Notice how the algorithm successfully converted our raw incomes into `0.0` (Low), `1.0` (Medium), and `2.0` (High) mathematically!)*

# 4. The Trade-off: Loss of Information
Binning is not a magic bullet. When you group a 22-year-old and a 35-year-old into the same "Young Adult" bucket, the machine learning model can no longer see the 13-year difference between them. To the model, they are identical. 

**The Golden Rule:** Only use binning if the continuous numbers are too noisy, heavily skewed by outliers, or if there is a clear non-linear threshold (e.g., people under 18 cannot legally sign a contract, so distinguishing between 16 and 17 doesn't matter, but distinguishing between 17 and 18 changes everything).

---

## Real-World Use Case or Analogy:
Think of Binning like **Grading a Final Exam**:

* **Continuous Data (The Raw Score)**: You take a 100-question test and get an 89. Your friend gets a 91. Technically, your friend scored 2 points higher than you. 
* **The Problem**: If a university admissions algorithm looks at raw scores across 10,000 different schools, the tiny variations (89 vs 91) might just be due to one teacher grading slightly harder than another. It's too noisy.
* **Binning (The Letter Grade)**: The school applies a `pd.cut` logic:
    * 90 - 100 = "A"
    * 80 - 89 = "B"
* **The Result**: You receive a B, and your friend receives an A. The tiny 2-point difference was caught on a critical boundary, while a student with an 81 and a student with an 89 are grouped together perfectly as "B" students. The university can now make sweeping decisions based on Letter Grades without over-analyzing single points.

---